In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import glob
# from upsetplot import from_contents, plot
from matplotlib import pyplot

from pathlib import Path

from dfmodel import DigitalFamilyBinary
from networkx.classes import non_neighbors

In [ ]:
design_matrix_test = pd.read_csv(
    "../0_data/design_matrix_test.tsv",
    sep='\t'
)

In [ ]:
design_matrix = pd.read_csv(
    "../0_data/design_matrix_train.tsv",
    sep='\t'
)

In [ ]:
design_matrix['survival_days'] = design_matrix['survival_days'].fillna(design_matrix['survival_days'].max())

In [ ]:
design_matrix_test['survival_days'] = design_matrix_test['survival_days'].fillna(design_matrix_test['survival_days'].max())

In [ ]:
design_matrix['organ_dysfunction'] = ((design_matrix[['sofa_score_increase_day1', 'sofa_score_increase_day2', 'sofa_score_increase_day3']] > 1).any(axis=1)).astype(int)
design_matrix_test['organ_dysfunction'] = ((design_matrix_test[['sofa_score_increase_day1', 'sofa_score_increase_day2', 'sofa_score_increase_day3']] > 1).any(axis=1)).astype(int)

In [ ]:
design_matrix.shape

In [ ]:
clinical_columns = [
    "sao2_ambulance",
    "resp_frequency_ambulance",
    "heart_rate_ambulance",
    "syst_bp_ambulance",
    "map_ambulance",
    "mental_status_ambulance",
    "temperature_ambulance",
    "crea_emergency_department",
    "bili_emergency_department",
    "crp_day1",
    "trc_emergency_department",
    "sex",
    "age"
]

continuous_clinical_columns = [
    "sao2_ambulance",
    "resp_frequency_ambulance",
    "heart_rate_ambulance",
    "syst_bp_ambulance",
    "map_ambulance",
    "temperature_ambulance",
    "crea_emergency_department",
    "bili_emergency_department",
    "trc_emergency_department",
    "crp_day1",
    "age"
]

categorical_clinical_columns = [
    "mental_status_ambulance",
]

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelBinarizer, MinMaxScaler
from sklearn.impute import SimpleImputer

training_data = design_matrix.copy()
testing_data = design_matrix_test.copy()

clinical_scaler = StandardScaler()

clinical_imputer = SimpleImputer()
min_max_scaler = MinMaxScaler()
sex_scaler = MinMaxScaler((0, 1))
constant_imputer = SimpleImputer(strategy="constant", fill_value=1.0)

training_data[continuous_clinical_columns] = clinical_imputer.fit_transform(training_data[continuous_clinical_columns])
training_data[["mental_status_ambulance"]] = constant_imputer.fit_transform(training_data[["mental_status_ambulance"]])

training_data[continuous_clinical_columns] = clinical_scaler.fit_transform(training_data[continuous_clinical_columns])
training_data[['sex']] = sex_scaler.fit_transform(training_data[['sex']])
#training_data[clinical_columns] = min_max_scaler.fit_transform(training_data[clinical_columns])

testing_data[continuous_clinical_columns] = clinical_imputer.transform(testing_data[continuous_clinical_columns])
testing_data[["mental_status_ambulance"]] = constant_imputer.transform(testing_data[["mental_status_ambulance"]])

testing_data[continuous_clinical_columns] = clinical_scaler.transform(testing_data[continuous_clinical_columns])
testing_data[['sex']] = sex_scaler.transform(testing_data[['sex']])
#testing_data[clinical_columns] = min_max_scaler.transform(testing_data[clinical_columns])

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
components = pca.fit_transform(training_data[clinical_columns])

training_data['PCA 1'] = components[:, 0]
training_data['PCA 2'] = components[:, 1]

In [ ]:
sns.scatterplot(
    data=training_data,
    x='PCA 1',
    y='PCA 2',
    hue="sepsis_or_septic_shock"
)

In [ ]:
from sklearn.cluster import OPTICS

optcs_model = OPTICS(
    min_samples=10,
)

optcs_model.fit(training_data[clinical_columns])

In [ ]:
optics_results = pd.DataFrame(
    {
        "sample": training_data.index.values,
        "reachability": optcs_model.reachability_[optcs_model.ordering_],
        "label": optcs_model.labels_[optcs_model.ordering_],
    }
)

In [ ]:
optics_results

In [ ]:
optics_results['label'].value_counts()

In [ ]:
sns.scatterplot(
    data=optics_results,
    x="sample",
    y="reachability",
)

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

class ConsensusClustering:

    def __init__(self, n_clusters=2, random_state=0, n_runs=20, subsample=0.9):
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.n_runs = n_runs
        self.subsample = subsample

    def fit_transform(self, X: pd.DataFrame):

        self.coassoc_matrix = np.zeros((X.shape[0], X.shape[0]))

        for run in range(self.n_runs):

            X_ = X.sample(frac=self.subsample, replace=False)

            kmeans = KMeans(n_clusters=self.n_clusters, random_state=self.random_state)

            labels = kmeans.fit_predict(X_)

            for i, idxi in enumerate(X_.index):

                for j, idxj in enumerate(X_.index):

                    if labels[i] == labels[j]:

                        self.coassoc_matrix[idxi, idxj] += 1

        self.coassoc_matrix /= self.n_runs

        linked = linkage(1 - self.coassoc_matrix, method="average")

        return fcluster(linked, t=self.n_clusters, criterion="maxclust")



In [ ]:
from scipy.stats import ecdf
from statsmodels.distributions.empirical_distribution import ECDF

clusterer_results = dict()

cdfs = dict()

range_n_clusters = range(2, 11)

for n_cluster in range_n_clusters:

    consensus_clusterer = ConsensusClustering(
        n_clusters=n_cluster,
        n_runs=100,
    )

    training_data[f'Consensus Labels{n_cluster}'] = consensus_clusterer.fit_transform(training_data[clinical_columns])

    clusterer_results[n_cluster] = consensus_clusterer

    consensus_values = consensus_clusterer.coassoc_matrix[np.triu_indices_from(consensus_clusterer.coassoc_matrix, k=1)]

    cdfs[n_cluster] = (ECDF(consensus_values)(np.arange(0.0, 1.0, 0.01)), np.arange(0.0, 1.0, 0.01), consensus_values)#ecdf(consensus_values)


In [ ]:
fig, ax = plt.subplots()

plot_data = pd.DataFrame()

for k, cdf in cdfs.items():

    sns.lineplot(
        x=cdf[1],
        y=cdf[0],
        ax=ax,
        label=k,
        drawstyle="steps-post",
    )

    temp_df = pd.DataFrame(
        {
            "Consensus Value": cdf[1],
            "ECDF": cdf[0],
            "K": [k for _ in range(len(cdf[0]))],
        }
    )

    plot_data = pd.concat([plot_data, temp_df])



In [ ]:

auc_data = []

for k, cdf in cdfs.items():

    auc_data.append(np.trapz(cdf[0], cdf[1]))

In [ ]:
auc_deltas = []

for left, right in zip(auc_data[:-1], auc_data[1:]):

    auc_delta = right - left

    auc_deltas.append(auc_delta)



In [ ]:
auc_plot_data = pd.DataFrame(
    {
        "AUC": auc_deltas,
        "K": range(len(auc_deltas)),
    }
)

In [ ]:
sns.lineplot(
    data=auc_plot_data,
    x="K",
    y="AUC",
)

In [ ]:
for k in range_n_clusters:

    sns.clustermap(
        data=clusterer_results[k].coassoc_matrix,
        cmap="Blues",

    )

In [ ]:
plot_data = pd.DataFrame()

for k, cdf in cdfs.items():

    temp_df = pd.DataFrame(
        {
            "Consensus Values": cdf[2],
            "K": [k for _ in range(len(cdf[2]))],
        }
    )

    plot_data = pd.concat([plot_data, temp_df])

In [ ]:
plot_data

sns.kdeplot(
    data=plot_data,
    x="Consensus Values",
    hue="K",
)

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []

for k in range_n_clusters:

    col = f"Consensus Labels{k}"

    silhouette_scores.append(silhouette_score(training_data[clinical_columns], training_data[col]))


In [ ]:
sns.lineplot(
    x=range_n_clusters,
    y=silhouette_scores,
)

In [ ]:
sns.scatterplot(
    data=training_data,
    x="PCA 1",
    y="PCA 2",
    hue="Consensus Labels4"
)

In [ ]:
for col in clinical_columns:

    print(col)
    sns.scatterplot(
        data=training_data,
        x="PCA 1",
        y="PCA 2",
        hue=col
    )
    plt.show()

In [ ]:

col = f"Consensus Labels4"

cluster_names = training_data[col].unique()

In [ ]:
cluster_centroids = dict()

fig, axs = plt.subplots(
    nrows=1,
    ncols=4,
    layout="constrained",
)

for i, cluster_name in enumerate(cluster_names):

    subset_training_data = training_data[training_data[col] == cluster_name].copy()

    cluster_centroids[cluster_name] = subset_training_data[clinical_columns].mean()

    sns.boxplot(
        data=subset_training_data[clinical_columns],
        ax=axs[i]
    )

    axs[i].set_title(cluster_name)

    axs[i].set_xticklabels(axs[i].get_xticklabels(), rotation=90)

fig.set_size_inches((20, 4))

In [ ]:
long = training_data.melt(
    id_vars=[col],    value_vars=clinical_columns,

)
long[col] = long[col].astype("category")

In [ ]:
long

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(
)
sns.boxplot(
    data=long,
    x="variable",
    y="value",
    hue="Consensus Labels4",
    ax=axs
)

#axs.set_xticklabels(axs.get_xticklabels(), rotation=90)

fig.set_size_inches((20, 4))

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(
)
sns.pointplot(
    data=long,
    y="variable",
    x="value",
    hue="Consensus Labels4",
    ax=axs
)

#axs.set_xticklabels(axs.get_xticklabels(), rotation=90)

fig.set_size_inches((4, 10))

In [ ]:

sns.scatterplot(
    data=training_data,
    x="PCA 1",
    y="PCA 2",
    hue=col
)
plt.show()

In [ ]:
centroids = pd.DataFrame.from_dict(cluster_centroids, orient="index")

In [ ]:
centroid_locations = pca.transform(centroids)

In [ ]:

sns.scatterplot(
    data=training_data,
    x="PCA 1",
    y="PCA 2",
    hue=col
)

sns.scatterplot(
    x=centroid_locations[:, 0],
    y=centroid_locations[:, 1],
    c="red",
    marker="X",
    s=200
)

plt.show()

In [ ]:
from sklearn.metrics import silhouette_samples

training_data['Silhoutte Samples'] = silhouette_samples(
    training_data[clinical_columns],
    labels=training_data["Consensus Labels4"],
)




In [ ]:
training_data = training_data.sort_values(
    by=["Consensus Labels4", "Silhoutte Samples"],
    ascending=[True, True]
)

In [ ]:
training_data['X_axis'] = np.arange(training_data.shape[0])

In [ ]:
sns.scatterplot(
    data=training_data,
    y="X_axis",
    x="Silhoutte Samples",
    hue="Consensus Labels4",
)

In [ ]:
sns.histplot(
    data=training_data,
    x="Silhoutte Samples",
)

In [ ]:
sns.scatterplot(
    data=training_data,
    x="PCA 1",
    y="PCA 2",
    hue="survival_days"
)
plt.show()

In [ ]:
from fa2_modified import ForceAtlas2
import matplotlib.pyplot as plt
import networkx as nx

fig, axs = plt.subplots(
    nrows=2,
    ncols=7,

)

axs = axs.flatten()

all_positions = []

for i, (n_neighbor, G) in enumerate(zip(n_neighbors, graphs)):

    forceatlas2 = ForceAtlas2(
        # Behavior alternatives
        outboundAttractionDistribution=True,  # Dissuade hubs
        linLogMode=False,  # NOT IMPLEMENTED
        adjustSizes=False,  # Prevent overlap (NOT IMPLEMENTED)
        edgeWeightInfluence=1.0,

        # Performance
        jitterTolerance=1.0,  # Tolerance
        barnesHutOptimize=True,
        barnesHutTheta=1.2,
        multiThreaded=False,  # NOT IMPLEMENTED

        # Tuning
        scalingRatio=2.0,
        strongGravityMode=False,
        gravity=1.0,

        # Log
        verbose=True)

    positions = forceatlas2.forceatlas2_networkx_layout(G, pos=None, iterations=1000)

    all_positions.append(positions)

    testing_data[f"{n_neighbors} Positions"] = positions

    nx.draw_networkx_nodes(G, positions, node_size=40, node_color="#1E555C", alpha=0.8, ax=axs[i])
    nx.draw_networkx_edges(G, positions, edge_color="gray", alpha=0.05, ax=axs[i])

    axs[i].axis('off')
    axs[i].set_title(f"{n_neighbor} Neighbors")

fig.set_size_inches((20, 7))
plt.tight_layout()

In [ ]:
fig.savefig(
    "digital_family_models_n_neighbors.pdf", dpi=300, bbox_inches="tight"
)

In [ ]:
from joblib import dump

with open("all_positions.pkl", "wb") as f:
    dump(all_positions, f)

In [ ]:
from matplotlib import pyplot as plt

fig, ax = plt.subplots()

nx.draw_networkx_nodes(graphs[2], all_positions[2], node_size=70, node_color="#1E555C", alpha=0.8, ax=ax)
nx.draw_networkx_edges(graphs[2], all_positions[2], edge_color="gray", alpha=0.05, ax=ax)

fig.set_size_inches((10, 10))
ax.axis('off')
plt.show()

In [ ]:
fig.savefig("graph.pdf", dpi=300, bbox_inches="tight")